# 04 - RQ3: Full vs. Reduced Feature Set

**Notebook version:** v1 -- 2026-07-28

Tests whether a SHAP-guided reduced feature set (~10 features, per the
RQ1 events-per-variable constraint) matches the full 591-feature model's
performance, without the two pitfalls flagged during synopsis review:

1. **Information leakage** - SHAP importance is computed fold-by-fold on
   training data only, never on the full dataset before splitting.
2. **Single-method bias** - an independent LASSO-based selection cross-checks
   the SHAP-selected set; agreement between the two is evidence of real signal.

See `docs/synopsis.docx`, "Solution to RQ3" for the full write-up, and
`src/feature_selection.py` for the underlying implementation.

In [ ]:
# --- Colab setup: run this cell first if you opened this notebook from GitHub in Colab ---
# If you're running locally in Jupyter from the notebooks/ folder, this cell does nothing.
# Safe to re-run: always anchors to /content so repeated runs never create nested clones.
# Also pulls latest changes and prints the commit hash, so you can confirm at a
# glance (against GitHub's commit history) that you're looking at the current version.
import os
import subprocess

try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

REPO_URL = "https://github.com/WJPsystems/secom-explainable-vm.git"
REPO_NAME = "secom-explainable-vm"

if IN_COLAB:
    os.chdir("/content")
    if not os.path.exists(REPO_NAME):
        !git clone "{REPO_URL}"
    else:
        os.chdir(f"/content/{REPO_NAME}")
        !git pull
        os.chdir("/content")
    os.chdir(f"/content/{REPO_NAME}/notebooks")
    !pip install -q -r ../requirements.txt

    commit_info = subprocess.run(
        ["git", "log", "-1", "--format=%h %ci"], capture_output=True, text=True
    ).stdout.strip()
    print(f"Colab setup complete. Working directory: {os.getcwd()}")
    print(f"Repo commit: {commit_info}")
    print("Compare this commit hash against GitHub's latest commit to confirm you're current.")
else:
    print("Not running in Colab -- assuming local Jupyter launched from the notebooks/ folder.")

In [ ]:
import sys

# Resolve src/ absolutely, independent of cell run order or current working
# directory -- works whether or not the Colab setup cell above has run yet.
try:
    import google.colab
    _SRC_PATH = "/content/secom-explainable-vm/src"
except ImportError:
    _SRC_PATH = "../src"
if _SRC_PATH not in sys.path:
    sys.path.append(_SRC_PATH)

import pandas as pd
from xgboost import XGBClassifier

from preprocessing import load_raw, screen_missingness, screen_variance, impute_median
from feature_selection import (
    nested_cv_shap_selection,
    lasso_cross_check,
    agreement_report,
)

X, y = load_raw()
X = impute_median(screen_variance(screen_missingness(X)))
print(f"Screened feature count: {X.shape[1]}")

## Step 1: Nested-CV SHAP feature selection

For each fold: cluster correlated sensors, fit the model on training data
only, compute SHAP on that same training data, and select the top-10
cluster representatives. Report how stable the selection is across folds -
this stability number is itself a finding worth including in the write-up.

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from artifacts import load_json

# Use RQ1's actual best tree ensemble architecture (not a hardcoded guess) --
# nested CV needs to retrain fresh per fold, so this rebuilds the same model
# TYPE RQ1 found best, rather than reusing RQ1's already-fitted instance.
rq1_summary = load_json("rq1_summary")
best_tree_name = rq1_summary["best_tree_model_name"]
pos_weight = (y == 0).sum() / (y == 1).sum()

if best_tree_name == "XGBoost":
    model_factory = lambda: XGBClassifier(
        n_estimators=200, max_depth=4, scale_pos_weight=pos_weight,
        eval_metric='logloss', random_state=42,
    )
elif best_tree_name == "RandomForest":
    model_factory = lambda: RandomForestClassifier(
        n_estimators=300, class_weight="balanced", random_state=42,
    )
else:
    raise ValueError(f"Unexpected best_tree_model_name from RQ1: {best_tree_name}")

print(f"Using RQ1's best tree ensemble architecture for nested-CV selection: {best_tree_name}")

result = nested_cv_shap_selection(X, y, model_factory, k=10, n_folds=5)

print("\nPer-fold selected features:")
for i, feats in enumerate(result['per_fold_features']):
    print(f"  Fold {i+1}: {feats}")

print("\nFeature stability (how many of 5 folds selected each feature):")
for feat, count in result['feature_stability'].most_common(15):
    print(f"  {feat}: {count}/5")


## Step 2: Build the final feature set and the watch list

The final ~10 features are the most stable across folds (not just the
single best fold). The watch list is the next-ranked 20-30 features that
narrowly missed the cut - reported alongside the model, not discarded.

In [ ]:
final_features = [feat for feat, _ in result['feature_stability'].most_common(10)]
watch_list = sorted(set(f for fold in result['watch_list_by_fold'] for f in fold))

print(f"Final feature set ({len(final_features)}): {final_features}")
print(f"\nWatch list ({len(watch_list)} candidates): {watch_list[:30]}")

## Step 3: Independent LASSO cross-check

A methodologically different feature-selection approach (L1-regularized
logistic regression). High overlap with the SHAP-selected set is
corroborating evidence; low overlap is a flag to investigate before
trusting the reduced set.

In [ ]:
lasso_features = lasso_cross_check(X, y)
print(f"LASSO-selected features ({len(lasso_features)}): {lasso_features[:20]}")

report = agreement_report(final_features, lasso_features[:10])
print(f"\nOverlap: {report['overlap']}")
print(f"SHAP-only: {report['shap_only']}")
print(f"LASSO-only: {report['lasso_only']}")
print(f"Jaccard similarity: {report['jaccard_similarity']:.2f}")

## Step 4: Full vs. reduced model comparison

Compare AUC-ROC (DeLong's test) and accuracy (Cohen's h / two-proportion
test) between the full 591-feature model and the reduced ~10-feature
model, plus McNemar's test on paired prediction disagreements.

In [ ]:
# TODO: fit full-feature and reduced-feature models on a common held-out
# test split, then apply DeLong's test, Cohen's h, and McNemar's test.
# See src/metrics.py for BER/MCC helpers used alongside these tests.

## Next steps

- Feed `final_features` into `05_attention_comparison_rq4.ipynb`
- Report feature stability, watch list, and LASSO overlap in the capstone write-up
- See `06_anomaly_safety_net.ipynb` for the exploratory full-feature-set safety net

## Export

Run the cell below last to export this notebook to a standalone HTML file.

In [ ]:
# --- Export this notebook to HTML (run this cell last) ---
# Works whether opened live from GitHub in Colab or run locally in Jupyter.
import json
import subprocess

NOTEBOOK_NAME = "04_feature_reduction_rq3"

try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

export_path = f"{NOTEBOOK_NAME}.ipynb"

if IN_COLAB:
    # Colab's own notebook JSON isn't the same file as the clone on disk --
    # this pulls the live, currently-run state (including your outputs)
    # directly from the Colab frontend, so nothing gets missed.
    from google.colab import _message
    ipynb_content = _message.blocking_request('get_ipynb', timeout_sec=30)['ipynb']
    with open(export_path, 'w') as f:
        json.dump(ipynb_content, f)

html_output = f"{NOTEBOOK_NAME}.html"
result = subprocess.run(
    ['jupyter', 'nbconvert', '--to', 'html', export_path, '--output', html_output],
    capture_output=True, text=True,
)
print(result.stdout)
if result.returncode != 0:
    print(result.stderr)
else:
    print(f"Exported to {html_output}")

if IN_COLAB and result.returncode == 0:
    from google.colab import files
    files.download(html_output)
